# Scam Detection Dataset Processing Pipeline

## Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


## Install Required Libraries

In [ ]:
!pip install -q pandas numpy tqdm scikit-learn

Libraries installed!


## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported!")

Libraries imported!


## Paths configuration


In [ ]:

parent_dir="/content/drive/MyDrive/scam_detection"
INPUT_FOLDER = f'{parent_dir}/ScamContent-v5/'

OUTPUT_FOLDER = f'{parent_dir}/ScamContent-v5-cleaned/'

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Input folder: {INPUT_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")

Input folder: /content/drive/MyDrive/scam_detection/ScamContent-v5/
Output folder: /content/drive/MyDrive/scam_detection/ScamContent-v5-cleaned/


##  Discover All CSV Files

In [ ]:
# Find all CSV files in the input folder
csv_files = glob.glob(os.path.join(INPUT_FOLDER, '*.csv'))

print(f"Found {len(csv_files)} CSV files")


Found 646 CSV files


In [ ]:
def clean_dataframe(df, filename):
    """
    Clean and normalize a single dataframe according to the rules.

    Args:
        df: Input dataframe
        filename: Name of the source file (for logging)

    Returns:
        Cleaned dataframe with columns:  text
    """
    initial_rows = len(df)

    df = df.copy() # a copy is made to avoid modifying the original data

    df.columns = df.columns.str.lower() # Convert column names to lowercase

    if 'text'  not in df.columns:
        print(f" Warning: {filename} missing 'text' column. Skipping.")
        return pd.DataFrame(columns=[ 'text']), initial_rows, 0, initial_rows

    # Remove rows with empty text first
    df = df[df['text'].notna()]
    df = df[df['text'].astype(str).str.strip() != '']

    # Keep only necessary columns
    df = df[['text']]

    # Reset index
    df = df.reset_index(drop=True)

    final_rows = len(df)
    dropped_rows = initial_rows - final_rows

    return df, initial_rows, final_rows, dropped_rows

print("Cleaning function defined!")

Cleaning function defined!


## Process All Files

This will load, clean, and combine all CSV files. This may take a few minutes depending on file sizes.

In [ ]:
# List to store all cleaned dataframes
all_dataframes = []

# Statistics
total_initial_rows = 0
total_final_rows = 0
total_dropped_rows = 0
files_processed = 0
files_failed = 0

print("Starting to process files...")
print("=" * 80)

# Process each file with progress bar
for file_path in tqdm(csv_files, desc="Processing files"):
    filename = os.path.basename(file_path)

    try:
        # Load CSV file
        df = pd.read_csv(file_path)

        # Clean the dataframe
        cleaned_df, initial, final, dropped = clean_dataframe(df, filename)

        # Add to statistics
        total_initial_rows += initial
        total_final_rows += final
        total_dropped_rows += dropped

        # Add to list if not empty
        if len(cleaned_df) > 0:
            all_dataframes.append(cleaned_df)
            files_processed += 1
        else:
            print(f"\n {filename}: No valid rows after cleaning")

    except Exception as e:
        print(f"\n Error processing {filename}: {str(e)}")
        files_failed += 1

print("\n" + "=" * 80)
print("PROCESSING SUMMARY:")
print("=" * 80)
print(f"Total files found: {len(csv_files)}")
print(f"Files successfully processed: {files_processed}")
print(f"Files failed: {files_failed}")
print(f"\nTotal rows (before cleaning): {total_initial_rows:,}")
print(f"Total rows (after cleaning): {total_final_rows:,}")
print(f"Total rows dropped: {total_dropped_rows:,}")
print(f"Retention rate: {(total_final_rows/total_initial_rows*100) if total_initial_rows > 0 else 0:.2f}%")
print("=" * 80)

Starting to process files...


Processing files: 100%|██████████| 646/646 [00:15<00:00, 40.71it/s] 


PROCESSING SUMMARY:
Total files found: 646
Files successfully processed: 646
Files failed: 0

Total rows (before cleaning): 68,141
Total rows (after cleaning): 68,141
Total rows dropped: 0
Retention rate: 100.00%


## Combine All Dataframes

In [ ]:
if len(all_dataframes) == 0:
    print("No data to combine! Check your input files and paths.")
else:
    print("Combining all dataframes...")

    # Concatenate all dataframes
    combined_df = pd.concat(all_dataframes, ignore_index=True)


    print(f"Combined dataset created with {len(combined_df):,} rows")

    # Remove duplicates (based on text)
    print("\n Removing duplicates...")
    before_dedup = len(combined_df)
    combined_df = combined_df.drop_duplicates(subset=['text'], keep='first')
    after_dedup = len(combined_df)

    print(f"Removed {before_dedup - after_dedup:,} duplicate rows")
    print(f"Final combined dataset: {after_dedup:,} rows")

    # Display sample rows
    print("\nSample rows from combined dataset:")
    display(combined_df.head(10))

Combining all dataframes...
Combined dataset created with 68,141 rows

 Removing duplicates...
Removed 17,482 duplicate rows
Final combined dataset: 50,659 rows

Sample rows from combined dataset:


,text
0,"After you know what you want, call our toll fr..."
1,You may also like Contact Us Today Free Gift W...
2,Her writing is good enough to have produced a ...
3,If Michigan residents have information regardi...
4,Mold Related Heath Risks Coughing & Sneezing s...
5,"Call us today at 425-289-3200 for a free, no-p..."
6,If you wish to speak with a representative of ...
7,Get a free case evaluation from us by calling ...
8,Giro d'Italia 2010: Cycling Weekly's coverage ...
9,Thank you for reading 5 articles this month* J...


## Split into Train (80%) and Test (20%) Sets

##  Generate Processing Report

In [ ]:
if 'combined_df' in locals():
    print("Saving combined dataset to Google Drive...")

    # Define output file path
    combined_output_path = os.path.join(OUTPUT_FOLDER, 'combined_scam_dataset.csv')

    # Save combined dataset
    combined_df.to_csv(combined_output_path, index=False)
    print(f"Combined dataset saved: {combined_output_path}")

    print("\n" + "=" * 80)
    print("FILE SAVED SUCCESSFULLY!")
    print("=" * 80)
    print(f"\nOutput location: {OUTPUT_FOLDER}")
    print(f"File created: combined_scam_dataset.csv - {len(combined_df):,} rows")
    print("=" * 80)
else:
    print("No combined_df available to save!")

Saving combined dataset to Google Drive...
Combined dataset saved: /content/drive/MyDrive/scam_detection/ScamContent-v5-cleaned/combined_scam_dataset.csv

FILE SAVED SUCCESSFULLY!

Output location: /content/drive/MyDrive/scam_detection/ScamContent-v5-cleaned/
File created: combined_scam_dataset.csv - 50,659 rows


In [ ]:
if 'combined_df' in locals():
    # Create a detailed processing report
    report = f"""
{'=' * 80}
SCAM DETECTION DATASET PROCESSING REPORT
{'=' * 80}

INPUT INFORMATION:
- Input folder: {INPUT_FOLDER}
- Total CSV files found: {len(csv_files)}
- Files successfully processed: {files_processed}
- Files failed: {files_failed}

DATA CLEANING STATISTICS:
- Initial total rows: {total_initial_rows:,}
- Rows after cleaning: {total_final_rows:,}
- Rows dropped during cleaning: {total_dropped_rows:,}
- Retention rate: {(total_final_rows/total_initial_rows*100) if total_initial_rows > 0 else 0:.2f}%

DEDUPLICATION:
- Rows before deduplication: {before_dedup:,}
- Rows after deduplication: {after_dedup:,}
- Duplicate rows removed: {before_dedup - after_dedup:,}

FINAL COMBINED DATASET:
- Total rows: {len(combined_df):,}

{'=' * 80}
PROCESSING COMPLETE!
{'=' * 80}
"""

    print(report)

    # Save report to file
    report_path = os.path.join(OUTPUT_FOLDER, 'processing_report.txt')
    with open(report_path, 'w') as f:
        f.write(report)

    print(f"\nProcessing report saved: {report_path}")


SCAM DETECTION DATASET PROCESSING REPORT

INPUT INFORMATION:
- Input folder: /content/drive/MyDrive/scam_detection/ScamContent-v5/
- Total CSV files found: 646
- Files successfully processed: 646
- Files failed: 0

DATA CLEANING STATISTICS:
- Initial total rows: 68,141
- Rows after cleaning: 68,141
- Rows dropped during cleaning: 0
- Retention rate: 100.00%

DEDUPLICATION:
- Rows before deduplication: 68,141
- Rows after deduplication: 50,659
- Duplicate rows removed: 17,482

FINAL COMBINED DATASET:
- Total rows: 50,659



PROCESSING COMPLETE!


Processing report saved: /content/drive/MyDrive/scam_detection/ScamContent-v5-cleaned/processing_report.txt
